In [ ]:
#PROJECT NAV BASICS
PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"
BQ_DS = "lending_doc_info"
PIPELINE_END_TABLE_1 = "financial_statements"
PIPELINE_END_TABLE_2 = "appraisals"

In [ ]:
%env ANTHROPIC_API_KEY="YOU-API-KEY"

In [ ]:
#PACKAGE IMPORTS
import os
import json
import pandas as pd
from google.cloud import bigquery
import anthropic

In [ ]:
#SETUP CLIENTS
bq_client = bigquery.Client(project=PROJECT_ID)
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [ ]:
# PULL STRUCTURED LOAN RECORDS ACROSS MULTIPLE BQ TABLES, MERGE INTO ONE JSON
def pull_loan_records(loan_number: str, table_ids: list[str]) -> dict | None:
    """
    Looks up a loan's structured records across multiple BQ tables and merges
    them into a single nested JSON, keyed by table (doc type):
        {
          "loan_number": ...,
          "financial_statements": {
            "source_filename": ...,
            "fields": {"total_revenue": {"value": ..., "confidence": ...}, ...}
          },
          "appraisals": {
            "source_filename": ...,
            "fields": {"appraised_value": {...}, ...}
          }
        }
    Only returns None if the loan number isn't found in ANY of the tables —
    a miss in one table alone isn't fatal, since a question may only be
    answerable from a subset of doc types.
    """
    combined = {"loan_number": loan_number}
    found_any = False

    for table_id in table_ids:
        query = f"""
            SELECT *
            FROM `{table_id}`
            WHERE loan_number = @loan_number
            LIMIT 1
        """
        job_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("loan_number", "STRING", loan_number)
            ]
        )
        result = bq_client.query(query, job_config=job_config).to_dataframe()

        if result.empty:
            continue  # this table has no record for this loan — not fatal, keep going

        found_any = True
        row = result.iloc[0].to_dict()

        identifier_cols = {"source_filename", "loan_number"}
        confidence_cols = {c for c in row if c.endswith("_confidence")}
        field_cols = {
            c for c in row
            if c not in identifier_cols and f"{c}_confidence" in confidence_cols
        }

        fields = {
            field: {"value": row[field], "confidence": row[f"{field}_confidence"]}
            for field in field_cols
        }

        table_name = table_id.split(".")[-1]  # e.g. "financial_statements"
        combined[table_name] = {
            "source_filename": row.get("source_filename"),
            "fields": fields,
        }

    return combined if found_any else None

In [ ]:
#PULL LOAN NUMBER FROM NATURAL LANGUAGE QUESTION
def extract_loan_number(user_question: str) -> str | None:
    """
    Uses the LLM to identify the loan number referenced in a natural-language
    question. Returns the loan number string, or None if no loan number is
    mentioned. This is the simplest possible stand-in for semantic search —
    the user has to name the loan explicitly for now (matches the fallback
    plan: exact-identifier lookup now, true semantic search as a later upgrade).
    """
    system_prompt = (
        "Extract the loan number mentioned in the user's question. "
        "Loan numbers follow the format MTN-##### (e.g. MTN-10234). "
        "Return ONLY the loan number, exactly as written, with no other text. "
        "If no loan number is mentioned in the question, return exactly: NONE"
    )
 
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=50,
        system=system_prompt,
        messages=[{"role": "user", "content": user_question}],
    )
 
    result = response.content[0].text.strip()
    return None if result == "NONE" else result

In [ ]:
#GENERATE GROUNDED ANSWER
def generate_grounded_answer(user_question: str, loan_context: dict) -> str:
    """
    Takes a user's natural-language question and the structured loan context
    (from pull_loan_record) and returns an LLM-generated answer grounded
    strictly in that context, with confidence flags surfaced.
    """
    context_str = json.dumps(loan_context, indent=2, default=str)
 
    system_prompt = (
        "You help answer questions about commercial real estate loans. "
        "You will be given structured context extracted from a loan's financial "
        "statement, including a confidence score (0-100) for each field. "
        "Always answer strictly using the values given in the context — "
        "never state a number that isn't present in the context. "
        "If the person asks about a field that isn't present in the context, "
        "respond: \"I can't see a value for that field, I'm sorry.\" "
        "When you do answer with a number, mention its confidence score. "
        "If the confidence score for a field is below 90, explicitly flag "
        "that the value should be manually verified against the source "
        "document before being relied on."
    )
 
    user_message = (
        f"{user_question}\n\n"
        f"Grounded context to answer the question:\n{context_str}"
    )
 
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
 
    return response.content[0].text

In [ ]:
#FUNCTION TO ORCHESTRATE PARSE-PULL-ANSWER
def answer_question(user_question: str, table_id: str) -> str:
    """
    Full parse-pull-inject-answer flow, driven entirely by a natural-language
    question:
      1. Extract the loan number from the question (LLM)
      2. Pull the structured record for that loan number (BQ)
      3. Generate a grounded answer using that record as context (LLM)
    """
    loan_number = extract_loan_number(user_question)
    if loan_number is None:
        return (
            "I couldn't find a loan number in your question. "
            "Could you include the loan number you're asking about?"
        )

    loan_data = pull_loan_records(loan_number, table_ids)
    if loan_data is None:
        return f"I don't have any records for loan number {loan_number}."

    return generate_grounded_answer(user_question, loan_data)

In [ ]:
table_ids = [f"{PROJECT_ID}.{BQ_DS}.{PIPELINE_END_TABLE_1}", f"{PROJECT_ID}.{BQ_DS}.{PIPELINE_END_TABLE_2}"]

In [ ]:
question1 = "What was the net operating income for loan number MTN-10234?"
question2 = "What was estimated value and cap rate for loan number MTN-10234?"
question3 = "Tell me a bit about loan number MTN-10234, cite source for any info given."
print('q1')
print(answer_question(question1, table_ids))
print('q2')
print(answer_question(question2, table_ids))
print('q3')
print(answer_question(question3, table_ids))

In [ ]:
trap_question = "What was estimated value and cap rate for loan number MTN-54321?"
print(answer_question(trap_question, table_ids))